<a href="https://colab.research.google.com/github/AnimeshRaj1234/Deep_Learning/blob/main/Customer_Churn_Prediction_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Kaggle Dataset
      ↓
#Remove unnecessary columns
      ↓
#Encode categorical columns
      ↓
#X / y split
      ↓
#Train / Test split
      ↓
#StandardScaler
      ↓
#Convert to PyTorch tensors
      ↓
#DataLoader
      ↓
#PyTorch ANN
      ↓
#Training
      ↓
#Evaluation

In [367]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# from sklearn.preprocessing import LabelEncoder

In [368]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rjmanoj/credit-card-customer-churn-prediction")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'credit-card-customer-churn-prediction' dataset.
Path to dataset files: /kaggle/input/credit-card-customer-churn-prediction


In [369]:
path

'/kaggle/input/credit-card-customer-churn-prediction'

In [370]:
import os
print(os.listdir(path))

['Churn_Modelling.csv']


In [371]:
file_path = os.path.join(path,'Churn_Modelling.csv')

In [372]:
file_path

'/kaggle/input/credit-card-customer-churn-prediction/Churn_Modelling.csv'

In [373]:
df = pd.read_csv(file_path)

In [374]:
df

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [375]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [376]:
df.duplicated().sum() # total number of duplicates rows

np.int64(0)

In [377]:
df['Exited'].value_counts() #imbalance datasets

,count
Exited,
0,7963
1,2037


In [378]:
df['Gender'].value_counts() #fair distribution

,count
Gender,
Male,5457
Female,4543


In [379]:
df['Geography'].value_counts() #imbalance

,count
Geography,
France,5014
Germany,2509
Spain,2477


In [380]:
# drop colmns ['RowNumber','CustomerId', 'Surname']
df.drop(columns=['RowNumber','CustomerId', 'Surname'], inplace=True)

In [381]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [382]:
df.shape

(10000, 11)

In [383]:
# categorical colmns : geography, gender
df = pd.get_dummies(df,columns=['Geography','Gender'],drop_first=True,dtype=int)

In [384]:
df

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,0,0,0
1,608,41,1,83807.86,1,0,1,112542.58,0,0,1,0
2,502,42,8,159660.80,3,1,0,113931.57,1,0,0,0
3,699,39,1,0.00,2,0,0,93826.63,0,0,0,0
4,850,43,2,125510.82,1,1,1,79084.10,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,39,5,0.00,2,1,0,96270.64,0,0,0,1
9996,516,35,10,57369.61,1,1,1,101699.77,0,0,0,1
9997,709,36,7,0.00,1,0,1,42085.58,1,0,0,0
9998,772,42,3,75075.31,2,1,0,92888.52,1,1,0,1


In [385]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CreditScore        10000 non-null  int64  
 1   Age                10000 non-null  int64  
 2   Tenure             10000 non-null  int64  
 3   Balance            10000 non-null  float64
 4   NumOfProducts      10000 non-null  int64  
 5   HasCrCard          10000 non-null  int64  
 6   IsActiveMember     10000 non-null  int64  
 7   EstimatedSalary    10000 non-null  float64
 8   Exited             10000 non-null  int64  
 9   Geography_Germany  10000 non-null  int64  
 10  Geography_Spain    10000 non-null  int64  
 11  Gender_Male        10000 non-null  int64  
dtypes: float64(2), int64(10)
memory usage: 937.6 KB


In [386]:
X = df.drop(columns=['Exited'])
y = df['Exited']

In [387]:
X

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,0,0,0
1,608,41,1,83807.86,1,0,1,112542.58,0,1,0
2,502,42,8,159660.80,3,1,0,113931.57,0,0,0
3,699,39,1,0.00,2,0,0,93826.63,0,0,0
4,850,43,2,125510.82,1,1,1,79084.10,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,39,5,0.00,2,1,0,96270.64,0,0,1
9996,516,35,10,57369.61,1,1,1,101699.77,0,0,1
9997,709,36,7,0.00,1,0,1,42085.58,0,0,0
9998,772,42,3,75075.31,2,1,0,92888.52,1,0,1


In [388]:
y

,Exited
0,1
1,0
2,1
3,0
4,0
...,...
9995,0
9996,0
9997,1
9998,1


In [389]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [390]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# import torch

In [391]:
import torch


In [392]:
type(y)

pandas.core.series.Series

# convert into tensors

In [393]:
X_train = torch.tensor(X_train,dtype=torch.float32)
X_test = torch.tensor(X_test,dtype=torch.float32)

y_train = torch.tensor(y_train.to_numpy(),dtype=torch.float32)
y_test = torch.tensor(y_test.to_numpy(),dtype=torch.float32)

In [394]:
X_train

tensor([[ 0.3565, -0.6558,  0.3457,  ..., -0.5795, -0.5764,  0.9132],
        [-0.2039,  0.2949, -0.3484,  ...,  1.7257, -0.5764,  0.9132],
        [-0.9615, -1.4164, -0.6954,  ..., -0.5795,  1.7349,  0.9132],
        ...,
        [ 0.8650, -0.0854, -1.3894,  ..., -0.5795, -0.5764, -1.0950],
        [ 0.1593,  0.3900,  1.0397,  ..., -0.5795, -0.5764,  0.9132],
        [ 0.4707,  1.1506, -1.3894,  ...,  1.7257, -0.5764,  0.9132]])

In [395]:
y_train

tensor([0., 0., 1.,  ..., 1., 1., 0.])

In [396]:
X_train.shape

torch.Size([8000, 11])

# tensordataset and dataloader

In [397]:
from torch.utils.data import TensorDataset,DataLoader

In [398]:
train_dataset = TensorDataset(X_train,y_train)
test_dataset = TensorDataset(X_test,y_test)

In [399]:
len(train_dataset)

8000

In [400]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

# build model

In [401]:
import torch.nn as nn

class ChurnModel(nn.Module):
  def __init__(self,input_size):
    super().__init__()

    self.network = nn.Sequential(
        nn.Linear(input_size,64),
        nn.ReLU(),
        nn.Linear(64,32),
        nn.ReLU(),
        nn.Linear(32,1),
        nn.Sigmoid()
    )
  def forward(self,features):
    out = self.network(features)
    return out


In [402]:
model = ChurnModel(input_size=11)

criterion = nn.BCELoss()

optimizer = torch.optim.Adam(
    model.parameters(), # all weights and biases
    lr=0.001
)

In [403]:
model.network

Sequential(
  (0): Linear(in_features=11, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
  (5): Sigmoid()
)

In [404]:
for X_batch,y_batch in train_loader:
  print(y_batch.min())
  print(y_batch.max())
  y_pred = model(X_batch)
  print(y_pred.min())
  print(y_pred.max())
  break

tensor(0.)
tensor(1.)
tensor(0.4255, grad_fn=<MinBackward1>)
tensor(0.4694, grad_fn=<MaxBackward1>)


In [405]:
epochs = 50

for epoch in range(epochs):

  model.train()

  for X_batch,y_batch in train_loader:
    y_pred = model(X_batch)
    loss = criterion(y_pred,y_batch.unsqueeze(1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  print(f'Epoch [{epoch+1}/{epochs}], Loss : {loss.item():.4f}')

Epoch [1/50], Loss : 0.3506
Epoch [2/50], Loss : 0.2991
Epoch [3/50], Loss : 0.5154
Epoch [4/50], Loss : 0.2475
Epoch [5/50], Loss : 0.1894
Epoch [6/50], Loss : 0.3250
Epoch [7/50], Loss : 0.2710
Epoch [8/50], Loss : 0.2937
Epoch [9/50], Loss : 0.1882
Epoch [10/50], Loss : 0.3676
Epoch [11/50], Loss : 0.2954
Epoch [12/50], Loss : 0.2988
Epoch [13/50], Loss : 0.3543
Epoch [14/50], Loss : 0.5008
Epoch [15/50], Loss : 0.3067
Epoch [16/50], Loss : 0.2870
Epoch [17/50], Loss : 0.2016
Epoch [18/50], Loss : 0.5011
Epoch [19/50], Loss : 0.4084
Epoch [20/50], Loss : 0.3652
Epoch [21/50], Loss : 0.2151
Epoch [22/50], Loss : 0.3857
Epoch [23/50], Loss : 0.2338
Epoch [24/50], Loss : 0.3006
Epoch [25/50], Loss : 0.1371
Epoch [26/50], Loss : 0.2387
Epoch [27/50], Loss : 0.5449
Epoch [28/50], Loss : 0.2233
Epoch [29/50], Loss : 0.2931
Epoch [30/50], Loss : 0.2568
Epoch [31/50], Loss : 0.3747
Epoch [32/50], Loss : 0.4178
Epoch [33/50], Loss : 0.4248
Epoch [34/50], Loss : 0.4018
Epoch [35/50], Loss : 0

In [406]:
# model.parameters().shape

In [407]:
! pip install torchinfo

In [408]:
from torchinfo import summary
summary(model,input_size=(32,11))

Layer (type:depth-idx)                   Output Shape              Param #
ChurnModel                               [32, 1]                   --
├─Sequential: 1-1                        [32, 1]                   --
│    └─Linear: 2-1                       [32, 64]                  768
│    └─ReLU: 2-2                         [32, 64]                  --
│    └─Linear: 2-3                       [32, 32]                  2,080
│    └─ReLU: 2-4                         [32, 32]                  --
│    └─Linear: 2-5                       [32, 1]                   33
│    └─Sigmoid: 2-6                      [32, 1]                   --
Total params: 2,881
Trainable params: 2,881
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.09
Input size (MB): 0.00
Forward/backward pass size (MB): 0.02
Params size (MB): 0.01
Estimated Total Size (MB): 0.04

In [412]:
# evaluation code

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

total = 0
correct = 0
model.eval()
with torch.no_grad():

  for X_batch, y_batch in test_loader:
    # Move data to the same device as the model
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)

    outputs = model(X_batch)
    predictions = (outputs >= .5).float()

    y_true = y_batch.unsqueeze(1)

    correct += (predictions == y_true).sum().item()

    total += y_true.size(0)

accuracy = correct / total

print(f'Test Accuracy: {accuracy*100:.2f}')

Test Accuracy: 86.55
